In [0]:
# ============================================================
# NOTEBOOK   : silver_order_payments
# PURPOSE    : Clean and transform bronze.order_payments table
# SOURCE     : fincompliance_catalog.bronze.order_payments
# DESTINATION: fincompliance_catalog.silver.order_payments
# TRANSFORMATIONS:
#   - Deduplicate on order_id and payment_sequential
#   - Standardise payment_type to lowercase
#   - Round payment_value to 2 decimal places
#   - Classify payment_value into ranges
#   - Add is_instalment flag
#   - Calculate instalment_value per payment
#   - Drop bronze audit columns
#   - Add silver_updated_at audit column
# ============================================================

In [0]:
%run ../configs/config

In [0]:
authenticate_spark(spark)
set_catalog(spark)

In [0]:
from pyspark.sql.functions import (
    col,
    lower,
    trim,
    when,
    lit,
    round as spark_round,
    current_timestamp,
    avg,
    count,
    sum as spark_sum
)

In [0]:
# ============================================================
# CELL 5 - READ FROM BRONZE
# ============================================================

df_order_payments = spark.table(f"{BRONZE_DB}.order_payments")

print("=" * 60)
print("BRONZE.ORDER_PAYMENTS — Before Transformation")
print("=" * 60)
print(f"Total rows    : {df_order_payments.count():,}")
print(f"Total columns : {len(df_order_payments.columns)}")

print("\nColumn names and data types:")
for field in df_order_payments.schema.fields:
    print(f"  → {field.name:<40} : {field.dataType}")

print("\nSample data BEFORE transformation (3 rows):")
df_order_payments.show(3, truncate=True)

print("\nPayment type breakdown:")
df_order_payments \
    .groupBy("payment_type") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

print("\nPayment value statistics:")
df_order_payments \
    .select("payment_value", "payment_installments") \
    .summary("min", "max", "mean", "count") \
    .show()

In [0]:
# CELL 6 - DEDUPLICATION
# Composite primary key:
# order_id + payment_sequential TOGETHER = unique row
# One order can have multiple payment methods
# ============================================================

rows_before = df_order_payments.count()
print(f"Rows BEFORE deduplication : {rows_before:,}")

df_order_payments_silver = df_order_payments \
    .dropDuplicates(["order_id", "payment_sequential"])

rows_after = df_order_payments_silver.count()
print(f"Rows AFTER deduplication  : {rows_after:,}")

duplicates_removed = rows_before - rows_after
print(f"Duplicates removed        : {duplicates_removed:,}")

if duplicates_removed == 0:
    print("No duplicates found")
else:
    print(f"{duplicates_removed:,} duplicates removed")

In [0]:
# STANDARDISE PAYMENT TYPE
# Lowercase and trim payment_type
# Handle "not_defined" payment types
# ============================================================

# Show BEFORE
print("Payment types BEFORE standardisation:")
df_order_payments_silver \
    .groupBy("payment_type") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

# Apply standardisation
df_order_payments_silver = df_order_payments_silver \
    .withColumn(
        "payment_type",
        trim(lower(col("payment_type")))
    ) \
    .withColumn(
        "payment_type",
        when(
            col("payment_type") == "not_defined",
            "unknown"
        )
        .otherwise(col("payment_type"))
    )

# Show AFTER
print("Payment types AFTER standardisation:")
df_order_payments_silver \
    .groupBy("payment_type") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

print("Payment type standardisation complete")

In [0]:
# ROUND PAYMENT VALUE
# Round payment_value to 2 decimal places
# Monetary values must always be precise
# ============================================================

# Show BEFORE
print("Payment value BEFORE rounding (5 rows):")
df_order_payments_silver \
    .select("payment_value") \
    .show(5, truncate=False)

# Apply rounding
df_order_payments_silver = df_order_payments_silver \
    .withColumn(
        "payment_value",
        spark_round(col("payment_value"), 2)
    )

# Show AFTER
print("Payment value AFTER rounding (5 rows):")
df_order_payments_silver \
    .select("payment_value") \
    .show(5, truncate=False)

# Check for zero payment values
zero_payments = df_order_payments_silver \
    .filter(col("payment_value") == 0.0) \
    .count()
print(f"Zero payment values found : {zero_payments:,}")

if zero_payments > 0:
    print("\nSample zero payment records:")
    df_order_payments_silver \
        .filter(col("payment_value") == 0.0) \
        .show(5, truncate=True)

print("Payment value rounding complete")

In [0]:
# HANDLE ZERO INSTALMENTS
# payment_installments = 0 is invalid
# Minimum should be 1 (single payment)
# Replace 0 with 1
# ============================================================

# Check zero instalments BEFORE
zero_instalments = df_order_payments_silver \
    .filter(col("payment_installments") == 0) \
    .count()
print(f"Zero instalments BEFORE : {zero_instalments:,}")

# Show sample of zero instalment records
if zero_instalments > 0:
    print("\nSample zero instalment records:")
    df_order_payments_silver \
        .filter(col("payment_installments") == 0) \
        .show(5, truncate=True)

# Replace 0 with 1
df_order_payments_silver = df_order_payments_silver \
    .withColumn(
        "payment_installments",
        when(
            col("payment_installments") == 0,
            lit(1)
        )
        .otherwise(col("payment_installments"))
    )

# Check zero instalments AFTER
zero_instalments_after = df_order_payments_silver \
    .filter(col("payment_installments") == 0) \
    .count()
print(f"Zero instalments AFTER  : {zero_instalments_after:,}")

# Show instalment distribution
print("\nInstalment distribution:")
df_order_payments_silver \
    .groupBy("payment_installments") \
    .count() \
    .orderBy("payment_installments") \
    .show(10, truncate=False)

print("Zero instalments handled")

In [0]:
# IS INSTALMENT FLAG
# Identify whether payment was made in instalments
# payment_installments > 1 → is_instalment = 1
# payment_installments = 1 → is_instalment = 0
# This column DID NOT EXIST in bronze
# ============================================================

# Add is_instalment flag
df_order_payments_silver = df_order_payments_silver \
    .withColumn(
        "is_instalment",
        when(col("payment_installments") > 1, 1)
        .otherwise(0)
    )

# Show results
print("Is instalment breakdown:")
df_order_payments_silver \
    .groupBy("is_instalment") \
    .count() \
    .orderBy("is_instalment") \
    .show(truncate=False)

# Show by payment type
print("Instalment usage by payment type:")
df_order_payments_silver \
    .groupBy("payment_type", "is_instalment") \
    .count() \
    .orderBy("payment_type", "is_instalment") \
    .show(truncate=False)

total = df_order_payments_silver.count()
instalment_count = df_order_payments_silver \
    .filter(col("is_instalment") == 1) \
    .count()
print(f"Instalment payments : {instalment_count:,} "
      f"({instalment_count/total*100:.1f}%)")
print(f"Single payments     : {total-instalment_count:,} "
      f"({(total-instalment_count)/total*100:.1f}%)")

print("Is instalment flag added")

In [0]:

# CALCULATE INSTALMENT VALUE
# How much does customer pay per month?
# instalment_value = payment_value / payment_installments
# This column DID NOT EXIST in bronze
# ============================================================

# Calculate instalment value
df_order_payments_silver = df_order_payments_silver \
    .withColumn(
        "instalment_value",
        spark_round(
            col("payment_value") /
            col("payment_installments"), 2
        )
    )

# Show results
print("Instalment value sample:")
df_order_payments_silver \
    .select(
        "order_id",
        "payment_type",
        "payment_value",
        "payment_installments",
        "instalment_value"
    ) \
    .show(10, truncate=True)

# Show statistics
print("\nInstalment value statistics:")
df_order_payments_silver \
    .select("instalment_value") \
    .summary("min", "max", "mean") \
    .show()

# Show average instalment by payment type
print("Average instalment value by payment type:")
df_order_payments_silver \
    .groupBy("payment_type") \
    .agg(
        avg("instalment_value").alias("avg_instalment"),
        avg("payment_installments").alias("avg_instalments"),
        avg("payment_value").alias("avg_total_value")
    ) \
    .orderBy("avg_total_value", ascending=False) \
    .show(truncate=False)

print("Instalment value calculated")

In [0]:
# ============================================================
# PAYMENT VALUE RANGE
# Classify payments into value ranges
# Low    : payment_value < 50
# Medium : payment_value 50 to 500
# High   : payment_value > 500
# This column DID NOT EXIST in bronze
# ============================================================

# Add payment value range
df_order_payments_silver = df_order_payments_silver \
    .withColumn(
        "payment_value_range",
        when(col("payment_value") < 50, "Low")
        .when(
            (col("payment_value") >= 50) &
            (col("payment_value") <= 500),
            "Medium"
        )
        .when(col("payment_value") > 500, "High")
        .otherwise("Unknown")
    )

# Show breakdown
print("Payment value range breakdown:")
df_order_payments_silver \
    .groupBy("payment_value_range") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

# Show by payment type
print("Payment value range by payment type:")
df_order_payments_silver \
    .groupBy("payment_type", "payment_value_range") \
    .count() \
    .orderBy("payment_type", "payment_value_range") \
    .show(truncate=False)

total = df_order_payments_silver.count()
print(f"\nTotal payments : {total:,}")
print("Payment value range added")

In [0]:
# DROP BRONZE AUDIT COLUMNS
# Remove ingestion_timestamp and source_file
# These belong to bronze layer only


# Show columns BEFORE dropping
print("Columns BEFORE dropping:")
print(f"Total columns : {len(df_order_payments_silver.columns)}")
for col_name in df_order_payments_silver.columns:
    print(f"  → {col_name}")

# Drop bronze audit columns
df_order_payments_silver = df_order_payments_silver \
    .drop("ingestion_timestamp", "source_file")

# Show columns AFTER dropping
print(f"\nColumns AFTER dropping:")
print(f"Total columns : {len(df_order_payments_silver.columns)}")
for col_name in df_order_payments_silver.columns:
    print(f"  → {col_name}")

print("Bronze audit columns dropped")

In [0]:
# ADD SILVER AUDIT COLUMN
# ============================================================

df_order_payments_silver = df_order_payments_silver \
    .withColumn("silver_updated_at", current_timestamp())

print("Final columns in silver.order_payments:")
print(f"Total columns : {len(df_order_payments_silver.columns)}")
for col_name in df_order_payments_silver.columns:
    print(f"  → {col_name}")

print(f"\nTotal rows : {df_order_payments_silver.count():,}")
print("Silver audit column added")

In [0]:
# WRITE TO SILVER


print("Writing to silver.order_payments...")

(
    df_order_payments_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_DB}.order_payments")
)

print(f"Written to {SILVER_DB}.order_payments")
print(f"Rows    : {df_order_payments_silver.count():,}")
print(f"Columns : {len(df_order_payments_silver.columns)}")

In [0]:
# CELL 16 - VERIFICATION
# ============================================================

print("=" * 60)
print("SILVER.ORDER_PAYMENTS VERIFICATION")
print("=" * 60)

df_silver = spark.table(f"{SILVER_DB}.order_payments")

# CHECK 1 — Row count
bronze_count = spark.table(
    f"{BRONZE_DB}.order_payments").count()
silver_count = df_silver.count()
print(f"\nCHECK 1 — Row counts:")
print(f"  Bronze : {bronze_count:,}")
print(f"  Silver : {silver_count:,}")
if bronze_count == silver_count:
    print(f"Row counts match")
else:
    print(f"Difference : {bronze_count - silver_count:,}")

# CHECK 2 — New columns
print(f"\nCHECK 2 — New columns:")
new_cols = [
    "is_instalment",
    "instalment_value",
    "payment_value_range",
    "silver_updated_at"
]
for c in new_cols:
    if c in df_silver.columns:
        print(f"{c:<30} exists")
    else:
        print(f"{c:<30} MISSING")

# CHECK 3 — Payment type breakdown
print(f"CHECK 3 — Payment type breakdown:")
df_silver \
    .groupBy("payment_type") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

# CHECK 4 — Payment value range
print(f"CHECK 4 — Payment value range:")
df_silver \
    .groupBy("payment_value_range") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

# CHECK 5 — Instalment breakdown
print(f"CHECK 5 — Instalment breakdown:")
df_silver \
    .groupBy("is_instalment") \
    .count() \
    .show(truncate=False)

# CHECK 6 — No not_defined payment types
not_defined = df_silver \
    .filter(col("payment_type") == "not_defined") \
    .count()
print(f"CHECK 6 — not_defined payment types : {not_defined}")
if not_defined == 0:
    print("No not_defined payment types")
else:
    print("not_defined still exists")

print("=" * 60)
print("silver.order_payments verification complete!")
print(f"Rows  : {silver_count:,}")
print(f"Cols  : {len(df_silver.columns)}")
print("=" * 60)

In [0]:

# ============================================================
# FIX - ADD MISSING instalment_value COLUMN
# ============================================================

# Read current silver table
df_order_payments_silver = spark.table(
    f"{SILVER_DB}.order_payments"
)

# Add missing instalment_value column
df_order_payments_silver = df_order_payments_silver \
    .withColumn(
        "instalment_value",
        spark_round(
            col("payment_value") /
            col("payment_installments"), 2
        )
    )

# Verify column added
print("Columns after fix:")
for col_name in df_order_payments_silver.columns:
    print(f"  → {col_name}")

# Write back to silver
(
    df_order_payments_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_DB}.order_payments")
)

print("\ninstalment_value column added successfully")

# Verify
df_check = spark.table(f"{SILVER_DB}.order_payments")
print(f"Total columns now : {len(df_check.columns)}")
print(f"Total rows        : {df_check.count():,}")

# Show sample
df_check.select(
    "payment_value",
    "payment_installments",
    "instalment_value"
).show(5, truncate=False)